In [26]:
import numpy as np 
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
 
%matplotlib widget

Here I aim to practice computational fluid techniques and compare different methods to visual stability and accuracy. 

## Advection

I am going to simulate the advection equation first in one and then two dimensions using  
- The first order upwind finite difference scheme
- The centered finite difference scheme
- The Godonov method (a conservative scheme)

I will do so by propogating a wave of gaussian form to the right.
I hope this will help me gain familiarity between these schemes and see their differences.

The one dimension advection equation is

$$ \frac{\partial u}{\partial t} + a \frac{\partial u}{\partial x} = 0$$

Where $a > 0$ is the wave speed

## First Order Upwind Finite Differece

Here we will be propogating rightwards and so the upwind scheme I should use will be explicitly

$$u^{n+1}_i = u^n_i - c(u^n_i - u^n_{i-1})$$

where $c = \frac{a}{\Delta x / \Delta t}$ is the courant number and influences the accuracy of the solver.

In [ ]:
# First Order Upwind Finte Difference

#-----------------
# Initial values
#----------------

L = 1
N = 100 # Number of Grid Cells
dx = L/N
dt = 0.008
a = 1
c = a/(dx / dt)
x0 = 0.3
sigma = 0.05

x = (np.arange(N) + 0.5) * dx # cell centers as domain

def initial_condition(x):
    return np.exp(-0.5 * ((x - x0) / sigma)**2)


u0 = initial_condition(x)

u_upwind = u0.copy()
u_centered = u0.copy()
u_godunov = u0.copy()

#---------------------
# Solvers
#---------------------
def exact_solution(t):
    # Periodic translation of the Gaussian.
    # Minimum-image distance handles the periodic boundary.
    distance = (x - a * t - x0 + L/2) % L - L/2
    return np.exp(-0.5 * (distance / sigma)**2)

def step_upwind(u):
    return u - c*(u - np.roll(u,1))

def step_centered(u):
    return u - 0.5*c*(np.roll(u,-1) - np.roll(u,1))

def step_godunov(u):
    outflux = a*u # flux going out to the right
    influx = a * np.roll(u,1) # flux coming in from the left
    u = u - (dt / dx)*(outflux - influx)
    return u

# --------------------------------------------------
# Animation
# --------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(14, 4),
                         sharex=True, sharey=True)

methods = [
    ("A: Upwind", u_upwind),
    ("B: Centered + Euler", u_centered),
    ("C: Godunov", u_godunov),
]

lines = []
for ax, (name, u) in zip(axes, methods):
    ax.plot(x, u0, "k--", alpha=0.45, label="Initial")
    line, = ax.plot(x, u, lw=2, label="Numerical")
    ax.set_title(name)
    ax.set_xlabel("x")
    ax.set_xlim(0, L)
    ax.set_ylim(-0.5, 1.5)
    ax.grid(alpha=0.25)
    lines.append(line)

axes[0].set_ylabel("u(x,t)")
time_text = fig.suptitle("")
fig.tight_layout()

step_number = 0
steps_per_frame = 2

def update(frame):
    global u_upwind, u_centered, u_godunov, step_number

    for _ in range(steps_per_frame):
        u_upwind = step_upwind(u_upwind)
        u_centered = step_centered(u_centered)
        u_godunov = step_godunov(u_godunov)
        step_number += 1

    t = step_number * dt
    us = [u_upwind, u_centered, u_godunov]

    for line, u in zip(lines, us):
        line.set_ydata(u)

    time_text.set_text(f"Linear advection: t = {t:.3f}")
    return lines + [time_text]

ani = FuncAnimation(
    fig, update, frames=100,
    interval=40, blit=False
)

plt.show()

## Godunov Method for Non-Linear Advection

Here I want to apply the Gudonov method to the advection equation once again but this time $a = a(x)$ and thus our PDE is no longer linear.
The conservative aspects of Godunov should support this.
The Godunov method is conservative in that it is dependent on the flux of the computed quantity into and out of a discrete cell.
In this way, our iteration looks like

$$\tilde u^{n+1}_i = \tilde u^n_i - \frac{\Delta t}{\Delta x} \left( F_{i + 1/2} - F_{i-1/2} \right)$$
where
$$F_{i + 1/2} = a \tilde u_i \hspace{1cm} F_{i-1/2} = a \tilde u_{i-1}$$

I now apply the Godunov method to the Inviscid Burger's Equation

$$ u_t + f(u)_x = 0, \hspace{5mm} f(u) = \frac{1}{2}u^2$$

Here, we must be careful to account for the shock and rarefaction waves that will occur now.